In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [4]:
X_train = pd.read_csv('data/X_train.csv',index_col='ROW_ID')
X_test = pd.read_csv('data/X_test.csv',index_col='ROW_ID')

y_train = pd.read_csv('data/y_train.csv',index_col='ROW_ID')
sample_submission = pd.read_csv('data/sample_submission.csv',index_col='ROW_ID')

# Features

In [5]:
RET_features = [f'RET_{i}' for i in range(1,20)]
SIGNED_VOLUME_features = [f'SIGNED_VOLUME_{i}' for i in range(1,20)]
TURNOVER_features = ['AVG_DAILY_TURNOVER']

In [6]:
for i in [3,5,10,15,20]:
    X_train[ f'AVERAGE_PERF_{i}'] = X_train[RET_features[:i+1]].mean(1)
    X_train[ f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_train.groupby('TS')[ f'AVERAGE_PERF_{i}'].transform('mean')
    
    X_test[ f'AVERAGE_PERF_{i}'] = X_test[RET_features[:i+1]].mean(1)
    X_test[ f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_test.groupby('TS')[ f'AVERAGE_PERF_{i}'].transform('mean')

In [7]:
features = RET_features + SIGNED_VOLUME_features + TURNOVER_features
features = features + [ f'AVERAGE_PERF_{i}' for i in [3,5,10,15,20]]
features = features + [ f'ALLOCATIONS_AVERAGE_PERF_{i}' for i in [3,5,10,15,20]]

In [8]:
ret_cols = [f"RET_{i}" for i in range(1, 21)]
r = X_train[ret_cols].to_numpy()

rtest = X_test[ret_cols].to_numpy()

def ema(arr, L):
    alpha = 2/(L+1)
    w = (1-alpha) ** np.arange(L)  # 0..L-1
    w = w / w.sum()
    return (arr[:, :L] * w).sum(axis=1)

X_train["ema3"]  = ema(r, 3)
X_train["ema5"]  = ema(r, 5)
X_train["ema10"] = ema(r,10)


X_test["ema3"]  = ema(rtest, 3)
X_test["ema5"]  = ema(rtest, 5)
X_test["ema10"] = ema(rtest,10)

m20 = r.mean(axis=1)
s20 = r.std(axis=1, ddof=0)

m20test = rtest.mean(axis=1)
s20test = rtest.std(axis=1, ddof=0)

X_train["z20"] = m20 / (s20 + 1e-12)
X_test["z20"] = m20test / (s20test + 1e-12)

In [51]:
sign = np.sign(r)  
signtest = np.sign(rtest)

def last_streak_len(sig_row, positive=True):
    # part de RET_1 vers RET_20
    target = 1 if positive else -1
    cnt = 0
    for v in sig_row[:20]:  # [:20] explicite
        if v == target:
            cnt += 1
        else:
            break
    return cnt

X_train["streak_pos"] = [last_streak_len(s, True) for s in sign]
X_train["streak_neg"] = [last_streak_len(s, False) for s in sign]

X_test["streak_pos"] = [last_streak_len(s, True) for s in signtest]
X_test["streak_neg"] = [last_streak_len(s, False) for s in signtest]

p = (r > 0).mean(axis=1)
p = np.clip(p, 1e-9, 1 - 1e-9)
X_train["sign_entropy20"] = -(p*np.log(p) + (1-p)*np.log(1-p))

ptest = (rtest > 0).mean(axis=1)    
ptest = np.clip(ptest, 1e-9, 1 - 1e-9)
X_test["sign_entropy20"] = -(ptest*np.log(ptest) + (1-ptest)*np.log(1-ptest))


# Pipeline

In [11]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

## Préparation des datasets

In [52]:

features = [c for c in X_train.columns if c not in ["TS","ALLOCATION"]] 

mu = X_train[features].mean()
sd = X_train[features].std().replace(0,1)
X_std = (X_train[features] - mu) / sd


df = pd.concat([X_train[["TS","ALLOCATION"]], X_std], axis=1)
df["y"] = y_train["target"]

X_all = torch.as_tensor(
    np.stack([g[features].to_numpy(np.float32) for _, g in df.groupby("TS")]),
    dtype=torch.float32
)
Y_all = torch.as_tensor(
    np.stack([g["y"].to_numpy(np.float32) for _, g in df.groupby("TS")]),
    dtype=torch.float32
)



## Création du SetTransformer

In [ ]:

class SetDataset(Dataset):
    def __init__(self, X, y ): self.X, self.y = X, y
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i], self.y[i]


class SAB(nn.Module):
    def __init__(self, d_in, d_model, n_heads=4, ff=256, p=0.1):
        super().__init__()
        self.proj = nn.Linear(d_in, d_model) if d_in != d_model else nn.Identity()
        self.mha  = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln1  = nn.LayerNorm(d_model)
        self.ffn  = nn.Sequential(nn.Linear(d_model, ff), nn.GELU(), nn.Linear(ff, d_model))
        self.ln2  = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(p)

    def forward(self, x ):
        h = self.proj(x)
        attn, _ = self.mha(h, h, h, need_weights=False)
        h = self.ln1(h + self.drop(attn))
        f = self.ffn(h)
        h = self.ln2(h + self.drop(f))
        return h

class SetTransformerPerElem(nn.Module):
    def __init__(self, d_in, d_model=256, n_heads=4, depth=3, ff=256, p=0.1):
        super().__init__()
        blocks = [SAB(d_in, d_model, n_heads, ff, p)]
        for _ in range(depth-1):
            blocks.append(SAB(d_model, d_model, n_heads, ff, p))
        self.blocks = nn.ModuleList(blocks)
        self.head   = nn.Linear(d_model, 1)

    def forward(self, x):
        h = x
        for blk in self.blocks:
            h = blk(h)
        y_cont = self.head(h).squeeze(-1)    
        return y_cont



## Entraînement

In [60]:
# TTS 

rng = np.random.RandomState(42)
B = X_all.shape[0]
idx = np.arange(B)
val_size = max(1, int(0.15 * B))
val_idx = rng.choice(idx, size=val_size, replace=False)
train_idx = np.setdiff1d(idx, val_idx)


ds_tr = SetDataset(X_all[train_idx], Y_all[train_idx])
ds_va = SetDataset(X_all[val_idx],   Y_all[val_idx])
dl_tr = torch.utils.data.DataLoader(ds_tr, batch_size=16, shuffle=True,  drop_last=False)
dl_va = torch.utils.data.DataLoader(ds_va, batch_size=32, shuffle=False, drop_last=False)



In [62]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SetTransformerPerElem(d_in=d_in, d_model=128, n_heads=4, depth=2, ff=256, p=0.1).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
loss_fn = nn.SmoothL1Loss(reduction="none") #nn.BCEWithLogitsLoss(reduction="none")

def train_epoch(dl):
    model.train()
    correct = 0
    nobs = 0
    for Xb, yb in dl:                 # mb: True = PAD
        Xb, yb = Xb.to(device), yb.to(device)
        y_pred = model(Xb)       # (B,N) continu

        loss_raw = loss_fn(y_pred, yb)    # (B,N)
        loss = loss_raw.mean()

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        with torch.no_grad():
            pred_sign = (y_pred >= 0).float()
            true_sign = (yb     >= 0).float()
            correct += (pred_sign == true_sign).sum().item()
            nobs+= yb.numel()
    return correct / max(nobs, 1)


In [61]:
@torch.no_grad()
def eval_epoch(dl):
    model.eval()
    correct, nobs, loss_sum = 0, 0, 0.0
    for Xb, yb in dl:
        Xb, yb = Xb.to(device), yb.to(device)
        y_pred = model(Xb)
        loss_sum += loss_fn(y_pred, yb).mean().item() * yb.size(0)  # moyenne pondérée par nb d'exemples (dates)
        pred_sign = (y_pred >= 0)
        true_sign = (yb     >= 0)
        correct += (pred_sign == true_sign).sum().item()
        nobs    += yb.numel()
        
    return (loss_sum / len(dl.dataset)), (correct / nobs)

In [63]:
best_acc = 0
best_epoch = 0
patience = 5
counter = 0
max_epochs = 100

for epoch in range(1, max_epochs+1):
    train_acc = train_epoch(dl_tr)
    val_loss, val_acc = eval_epoch(dl_va)

    improved = val_acc > best_acc + 1e-4
    if improved:
        best_acc = val_acc
        wait = 0
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    else:
        wait += 1

    print(f"Epoch {epoch:03d} | train_acc={train_acc*100:.2f}% | val_acc={val_acc*100:.2f}% | val_loss={val_loss:.6f} | {'*' if improved else ''}")

    if wait >= patience:
        print(f"Early stopping at epoch {epoch} (best val_acc={best_acc*100:.2f}%)")
        break

# Restaurer le meilleur modèle (val_acc max)
if best_state is not None:
    model.load_state_dict(best_state)


Epoch 001 | train_acc=49.93% | val_acc=48.75% | val_loss=0.000060 | *
Epoch 002 | train_acc=50.14% | val_acc=49.19% | val_loss=0.000014 | *
Epoch 003 | train_acc=50.02% | val_acc=49.01% | val_loss=0.000032 | 
Epoch 004 | train_acc=49.91% | val_acc=49.07% | val_loss=0.000041 | 
Epoch 005 | train_acc=50.05% | val_acc=50.96% | val_loss=0.000018 | *
Epoch 006 | train_acc=50.02% | val_acc=49.76% | val_loss=0.000003 | 
Epoch 007 | train_acc=49.97% | val_acc=51.02% | val_loss=0.000003 | *
Epoch 008 | train_acc=50.07% | val_acc=49.08% | val_loss=0.000004 | 
Epoch 009 | train_acc=49.74% | val_acc=49.13% | val_loss=0.000003 | 
Epoch 010 | train_acc=50.37% | val_acc=50.96% | val_loss=0.000003 | 
Epoch 011 | train_acc=49.88% | val_acc=49.03% | val_loss=0.000008 | 
Epoch 012 | train_acc=50.05% | val_acc=49.13% | val_loss=0.000003 | 
Early stopping at epoch 12 (best val_acc=51.02%)


## Inférence pour un batch (N, d_in) quelconque

In [ ]:
def predict_for_date(df_date_features: pd.DataFrame) -> pd.DataFrame:
    x = torch.tensor(df_date_features[features].to_numpy(np.float32))
    n = x.shape[0]
    if n < max_n:
        x_pad = torch.zeros((max_n, d_in), dtype=torch.float32); m = torch.ones((max_n,), dtype=torch.bool)
        x_pad[:n] = x; m[:n] = False
        x, m = x_pad.unsqueeze(0).to(device), m.unsqueeze(0).to(device)
    else:
        x, m = x.unsqueeze(0).to(device), torch.zeros((1, n), dtype=torch.bool, device=device)
    model.eval()
    with torch.no_grad():
        y_pred = model(x, mask=m).squeeze(0).cpu().numpy()[:n]
        pred = (y_pred >= 0).astype(int)
    out = df_date_features[["ALLOCATION"]].copy()
    out["pred"] = pred
    #out["pred"] = (out["proba"] >= 0.5).astype(int)
    return out